# 📓 Notebook 1｜紋理特徵：一階統計・共生矩陣・游程・Laws 遮罩

> 對應講義 **Part 2**（知識地圖站 1–3）
>
> 本筆記本全部使用**合成紋理**（不需要下載任何圖片），但方法完全等同課本 Fig 7.3 的實測流程。

## Step 0｜合成三種「個性」的紋理圖
- `smooth`：平滑圖（大範圍亮度緩慢變化）
- `coarse`：粗糙圖（grass 風，高對比、細碎）
- `periodic`：週期圖（規則條紋）
每張圖都是一個二維陣列，我們來看看它們長什麼樣。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage import data

rng = np.random.default_rng(7)
N = 128
noise = rng.standard_normal((N, N))

# 平滑：雜訊經過強高斯濾波 → 只剩大範圍變化
smooth  = ndimage.gaussian_filter(noise, sigma=6.0)
# 粗糙：小範圍變化、保留高頻細節
coarse  = ndimage.gaussian_filter(noise, sigma=1.2)
# 週期：棋盤格
rows, cols = np.indices((N, N))
periodic = np.sin(rows * 0.6) * np.sin(cols * 0.6) * 0.5 + 0.5

textures = {'平滑 smooth': smooth, '粗糙 coarse': coarse, '週期 periodic': periodic}
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, img) in zip(axes, textures.items()):
    ax.imshow(img, cmap='gray')
    ax.set_title(name)
    ax.axis('off')
plt.show()

## Step 1｜一階統計：自己寫一遍（課本 7.1–7.5）

把課本的公式一個一個翻譯成程式。口訣：**一階統計只問「亮度值」，不問「位置」**。

In [ ]:
def first_order_stats(img, Ng=16):
    """從影像直方圖計算一階統計特徵（課本 7.1–7.5）"""
    # 把亮度正規化到 [0, Ng-1]（課本 MATLAB 程式也是這麼做）
    img = (img - img.min()) / (img.max() - img.min() + 1e-12)
    g = np.round(img * (Ng - 1)).astype(int)
    # 直方圖 P(I)（機率）
    P, _ = np.histogram(g, bins=Ng, range=(0, Ng - 1), density=True)
    I = np.arange(Ng)
    m1 = (P * I).sum()                       # m1 = E[I] 平均亮度
    var = (P * (I - m1) ** 2).sum()          # μ2 = σ² 變異數
    sd = np.sqrt(var)
    skew = (P * (I - m1) ** 3).sum() / sd ** 3      # μ3 偏斜度
    kurt = (P * (I - m1) ** 4).sum() / var ** 2 - 3 # μ4 峰態（減 3 → 常態=0）
    ent = -(P[P > 0] * np.log2(P[P > 0])).sum()     # 熵 H（bits）
    return {'mean': m1, 'var': var, 'skew': skew, 'kurt': kurt, 'entropy': ent}

for name, img in textures.items():
    s = first_order_stats(img)
    print(f'{name:12s}  mean={s["mean"]:5.2f}  var={s["var"]:6.2f}  '
          f'skew={s["skew"]:6.3f}  kurt={s["kurt"]:7.2f}  entropy={s["entropy"]:5.2f}')

### ✏️ 練習
觀察上面三個紋理的 entropy：哪個最高？為什麼？
（提示：均勻直方圖 → 熵最大。粗糙圖的灰階是不是「最平均地」分布在 0–15？）

## Step 2｜二階統計：手寫共生矩陣（課本 7.6–7.11）

先用手寫程式驗證課本 Example 的 4×4 影像（講義互動 Demo 2 的同一張圖）。

In [ ]:
img4 = np.array([[0, 0, 2, 2],
                [1, 1, 0, 0],
                [3, 2, 3, 3],
                [3, 2, 2, 2]])

def cooccurrence(img, d=1, angle=0):
    """共生矩陣：angle 0=0°, 1=45°, 2=90°, 3=135°（只支援 d=1）"""
    Ng = int(img.max()) + 1
    M = np.zeros((Ng, Ng))
    delta = [(0, 1), (1, 1), (1, 0), (1, -1)][angle]
    H, W = img.shape
    for r in range(H):
        for c in range(W):
            for (dr, dc) in (delta, (-delta[0], -delta[1])):  # 正、反兩個方向
                r2, c2 = r + dr * d, c + dc * d
                if 0 <= r2 < H and 0 <= c2 < W:
                    M[img[r, c], img[r2, c2]] += 1
    return M / M.sum()   # 除以總對數 R → 機率

A0 = cooccurrence(img4, 1, 0)
np.set_printoptions(precision=3, suppress=True)
print('A0(d=1) 除以 1/24 後（對照課本 7.8）：')
print((A0 * 24).astype(int))
print('\n課本 7.8 印刷值：')
print(np.array([[4, 1, 1, 0], [1, 2, 0, 0], [1, 0, 6, 3], [0, 0, 3, 2]]))

In [ ]:
def haralick_features(M):
    """從共生矩陣算四大特徵（課本 7.8–7.11）"""
    Ng = M.shape[0]
    idx = np.arange(Ng)
    ASM = (M ** 2).sum()                                   # 角二階矩
    CON = 0
    for n in range(Ng):                                    # 對比度
        CON += n * n * M[np.abs(idx[:, None] - idx[None, :]) == n].sum()
    IDF = (M / (1 + (idx[:, None] - idx[None, :]) ** 2)).sum()  # 反差分矩
    H = -(M[M > 0] * np.log2(M[M > 0])).sum()              # 熵
    return ASM, CON, IDF, H

ASM, CON, IDF, H = haralick_features(A0)
print(f'| ASM 角二階矩={ASM:.4f} | CON 對比度={CON:.2f} | IDF={IDF:.4f} | 熵={H:.2f} |')

## Step 3｜用 skimage 一行算全部（工業標準做法）

現實世界沒人手寫共生矩陣——都用 `skimage.feature.graycomatrix` + `graycoprops`。
我們來「對答案」：手寫 vs 套件。

In [ ]:
from skimage.feature import graycomatrix, graycoprops

glcm = graycomatrix(img4, distances=[1], angles=[0], levels=4,
                    symmetric=True, normed=True)
print('套件的共生矩陣：')
print(glcm[:, :, 0, 0])

# skimage 0.19+ 的 graycoprops 一次只接受一個屬性名稱（回傳 [d, angle] 陣列）
print(f"套件 CON     = {graycoprops(glcm, 'contrast')[0,0]:.3f}      （我們手寫 {CON:.3f}）")
print(f"套件 Energy  = {graycoprops(glcm, 'energy')[0,0]:.3f}    （我們手寫 ASM {ASM:.3f}）")
# 注意：skimage 的 homogeneity 定義與課本 IDF 幾乎相同（分母 1+|i-j| vs 1+(i-j)²）：細節略異
print(f"套件 Homog   = {graycoprops(glcm, 'homogeneity')[0,0]:.3f}   （我們手寫 IDF {IDF:.3f}）")
print(f"套件 Correl  = {graycoprops(glcm, 'correlation')[0,0]:.3f}   （課本 Table 7.1 的 f3）")

> 🧠 手寫的目的只有一個：**看懂套件在算什麼**。看懂之後，職業生涯都用 `graycomatrix`（方向 × 距離全算，4 個方向平均 = 旋轉容忍）。

### ✏️ 練習
試著把 `angles=[0]` 改成 `angles=[0, 45, 90, 135]`，對 4 個方向取平均，再跟單方向的數值比較。

## Step 4｜游程特徵 Run-Length（課本 7.12–7.18）

對同一張 4×4 圖驗證課本 (7.12)(7.13) 的游程矩陣。

In [ ]:
def run_length_matrix(img, angle=0):
    """游程矩陣 Q_RL(i,j)：灰階 i 出現長度 j 的游程次數"""
    Ng = int(img.max()) + 1
    N = img.shape[0]
    Q = np.zeros((Ng, N), dtype=int)
    def scan(line):
        if not line: return
        val, length = img[line[0]], 1
        for p in line[1:]:
            if img[p] == val:
                length += 1
            else:
                Q[val, length - 1] += 1
                val, length = img[p], 1
        Q[val, length - 1] += 1
    H = W = img.shape[0]
    if angle == 0:      # 0°：每一列
        for r in range(H): scan([(r, c) for c in range(W)])
    elif angle == 1:    # 45°：對角線（r+c 固定）
        for s in range(2 * N - 1):
            scan([(r, s - r) for r in range(N) if 0 <= s - r < N])
    elif angle == 2:    # 90°：每一欄
        for c in range(W): scan([(r, c) for r in range(H)])
    else:               # 135°：反對角線（r-c 固定）
        for s in range(-(N - 1), N):
            scan([(r, r - s) for r in range(N) if 0 <= r - s < N])
    return Q

print('Q_RL(0°)（課本 7.12）')
print(run_length_matrix(img4, 0))
print('\nQ_RL(45°)（課本 7.13）')
print(run_length_matrix(img4, 1))

In [ ]:
def run_length_features(Q, n_pixels):
    """課本 7.14–7.18 的游程特徵"""
    total = Q.sum()
    jj = np.arange(1, Q.shape[1] + 1)
    SRE  = (Q / jj ** 2).sum() / total      # 短游程加權
    LRE  = (Q * jj ** 2).sum() / total      # 長游程加權
    GLNU = (Q.sum(axis=1) ** 2).sum() / total   # 灰階不均勻度
    RLN  = (Q.sum(axis=0) ** 2).sum() / total   # 游程長度不均勻度
    RP   = total / n_pixels                 # 游程比例
    return SRE, LRE, GLNU, RLN, RP

# 對課本 Fig 7.3 那種「平滑 vs 粗糙」場景：用我們的合成紋理
for name, img in textures.items():
    g = np.round(((img - img.min()) / (img.max() - img.min())) * 3).astype(int)
    sre, lre, glnu, rln, rp = run_length_features(run_length_matrix(g, 0), g.size)
    print(f'{name:12s} SRE={sre:6.3f}  LRE={lre:7.2f}  GLNU={glnu:7.1f}  RLN={rln:6.0f}  RP={rp:5.3f}')

### ✏️ 練習（重點！）
對照課本 Table 7.3：**平滑圖的 LRE 應該遠大於粗糙圖**。上面結果符合嗎？
如果不符，想想為什麼——（提示：我們的 `smooth` 是不是太「軟」了？把灰階壓到 4 階（`*3`）後，`smooth` 有沒有可能只剩一種灰階？）

## Step 5｜Laws 遮罩（課本 7.19）

9 個 mask = 3 條基本向量（平滑/邊緣/斑點）的兩兩乘積。用卷積掃過圖片 → 9 張特徵圖 → 從每張算一階統計。

In [ ]:
from scipy.ndimage import convolve

v = [np.array([1, 2, 1]), np.array([-1, 0, 1]), np.array([-1, 2, -1])]
names = ['L3 平滑', 'E3 邊緣', 'S3 斑點']
masks = []
for a in v:
    for b in v:
        masks.append(np.outer(a, b) / 9)   # 除以 9 讓能量可比較

img0 = np.clip((coarse - coarse.min()) / (coarse.max() - coarse.min()), 0, 1)
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for k, m in enumerate(masks):
    out = convolve(img0, m, mode='wrap')
    axes[k // 3, k % 3].imshow(out, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[k // 3, k % 3].set_title(f'{names[k // 3]}×{names[k % 3]}')
    axes[k // 3, k % 3].axis('off')
plt.suptitle('粗糙紋理 × 9 個 Laws 遮罩 → 9 張特徵圖', fontsize=13)
plt.show()

In [ ]:
# 從 9 張特徵圖各取「變異數」當特徵 → 9 維特徵向量
def laws_features(img):
    img = (img - img.min()) / (img.max() - img.min() + 1e-12)
    return np.array([convolve(img, m, mode='wrap').var() for m in masks])

print('每個紋理的 9 維 Laws 特徵（變異數）：')
for name, img in textures.items():
    f = laws_features(img)
    print(f'{name:12s} ' + ' '.join(f'{x:6.3f}' for x in f))

# 看一看：不同紋理的特徵向量是不是「長得不同」？（這就是可分類性的來源）
mat = np.vstack([laws_features(img) for img in textures.values()])
plt.figure(figsize=(8, 2.5))
plt.imshow(mat, cmap='turbo', aspect='auto')
plt.yticks(range(3), textures.keys())
plt.xlabel('Laws 遮罩編號 0–8')
plt.title('3 種紋理 × 9 個遮罩 → 特徵「指紋」')
plt.colorbar(shrink=0.8)
plt.show()

## 🏆 本筆記本小結（30 秒回顧）
| 你學會了 | 對應課本 | 業界版本 |
|---|---|---|
| 一階統計（直方圖 5 兄弟） | (7.1)–(7.5) | `np.histogram` |
| 共生矩陣 4 方向 | (7.6)–(7.11), Table 7.1 | `skimage.feature.graycomatrix/graycoprops` |
| 游程矩陣 + SRE/LRE | (7.12)–(7.18) | 手寫 or mahotas |
| Laws 9 遮罩 | (7.19) | CNN 卷積核的祖先 |